# SGD : déplacer le poids et recalculer la perte

Passer de la pente à une modification dont nous pouvons vérifier l’effet.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#une-iteration-entrainement).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Reprenons exactement L=(w−2)², au point w=0. Nous connaissons la pente −4. Le pas SGD soustrait le taux multiplié par cette pente.

| Quantité | Avant | Calcul du pas | Après |
| --- | --- | --- | --- |
| Poids w | 0 | 0 − 0,1×(−4) | 0,4 |
| Perte L | 4 | (0,4−2)² | 2,56 |

La perte diminue ici. Il faut recalculer la formule après la modification pour connaître cette nouvelle valeur.

torch.optim.SGD([w],lr=0.1) reçoit une liste de paramètres à régler et le taux. zero_grad remet à zéro les contributions anciennes ; backward calcule les nouvelles ; step applique le déplacement.

## Démonstration 1 · Faire le pas, puis regarder le nouveau point

Vérifier à quel moment chaque objet change.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch


### Choisir les paramètres et leur mise à jour

L’optimiseur reçoit w, et non sa valeur extraite par item(). Il doit conserver l’objet qu’il modifiera.

In [4]:
w = torch.tensor(0., requires_grad=True)
optimiseur = torch.optim.SGD([w], lr=0.1)


In [5]:
observer('w', w)

w forme () type torch.float32 requires_grad True
  valeurs : 0.0


### Remettre les gradients à zéro

Nous ne voulons pas additionner les dérivées d’un ancien lot indépendant.

In [6]:
optimiseur.zero_grad()


In [7]:
observer_entrainement({'w': w, 'optimiseur': optimiseur}, montrer_poids=False)

w.grad = None


### Calculer l’état suivant

La perte vaut 4 et le gradient −4. Nous gardons deux nombres pour comparer avant et après le pas.

In [8]:
perte = (w - 2.) ** 2


In [9]:
observer('perte', perte)

perte forme () type torch.float32 requires_grad True
  valeurs : 4.0


### Calculer les gradients

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [10]:
perte.backward()


In [11]:
observer_entrainement({'w': w, 'optimiseur': optimiseur, 'perte': perte}, montrer_poids=False)

w.grad forme () type torch.float32 requires_grad False
  valeurs : -4.0


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [12]:
gradient_avant = w.grad.item()
poids_avant = w.item()


In [13]:
observer('gradient_avant', gradient_avant)
observer('poids_avant', poids_avant)

gradient_avant = -4.0
poids_avant = 0.0


### Appliquer une mise à jour

step lit le gradient et soustrait 0,1×(−4). Le poids augmente jusqu’à 0,4.

In [14]:
optimiseur.step()


In [15]:
observer_entrainement({'w': w, 'optimiseur': optimiseur, 'perte': perte, 'gradient_avant': gradient_avant, 'poids_avant': poids_avant}, montrer_poids=True)

w forme () type torch.float32 requires_grad True
  valeurs : 0.4000000059604645
w.grad forme () type torch.float32 requires_grad False
  valeurs : -4.0


### Calculer l’état suivant

La nouvelle expression utilise le poids modifié. Nous obtenons 2,56, et non le coût 4 conservé dans perte.

In [16]:
nouvelle_perte = (w - 2.) ** 2


In [17]:
observer('nouvelle_perte', nouvelle_perte)

nouvelle_perte forme () type torch.float32 requires_grad True
  valeurs : 2.56000018119812


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [18]:
print("avant", poids_avant, perte.item(), "gradient", gradient_avant)
print("apres", round(w.item(), 6), round(nouvelle_perte.item(), 6))
# Même pente initiale, mais un taux trop grand.


avant 0.0 4.0 gradient -4.0
apres 0.4 2.56


### Calculer l’état suivant

Cette variante repart du même poids initial et de la même pente. Un taux 1,1 pousse le poids jusqu’à 4,4, au-delà du minimum.

In [19]:
taux_trop_grand = 1.1
autre_poids = poids_avant - taux_trop_grand * gradient_avant


In [20]:
observer('taux_trop_grand', taux_trop_grand)
observer('autre_poids', autre_poids)

taux_trop_grand = 1.1
autre_poids = 4.4


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [21]:
print("grand pas", round(autre_poids, 6), "perte", round((autre_poids - 2.) ** 2, 6))

grand pas 4.4 perte 5.76


Le taux décide de la longueur du déplacement, pas seulement de sa direction. Le grand pas augmente la perte malgré une pente initialement utile.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Après un pas SGD de taux 0,25 sur (w−2)², depuis w=0, quelle valeur prend w ?

In [22]:
exo_w = torch.tensor(0.,requires_grad=True)
exo_optimiseur = torch.optim.SGD([exo_w],lr=.25)
((exo_w-2.)**2).backward()
exo_optimiseur.step()
exo_apres = ...  # Remplacer les points par votre réponse.
if exo_apres is Ellipsis or exo_apres is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_apres)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

SGD soustrait taux×gradient : le gradient initial est −4.

</details>

### Exercice 2

En changeant seulement le taux à 0,5, quelle perte obtient-on après le pas ?

In [23]:
exo_w = torch.tensor(0.,requires_grad=True)
exo_optimiseur = torch.optim.SGD([exo_w],lr=.5)
((exo_w-2.)**2).backward()
exo_optimiseur.step()
exo_perte_apres = ...  # Remplacer les points par votre réponse.
if exo_perte_apres is Ellipsis or exo_perte_apres is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_perte_apres)-((exo_w-2.)**2).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le poids arrive à 2 dans cette fonction quadratique particulière. Cela ne se généralise pas à tout réseau.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_w = torch.tensor(0.,requires_grad=True)
exo_optimiseur = torch.optim.SGD([exo_w],lr=.25)
((exo_w-2.)**2).backward()
exo_optimiseur.step()
exo_apres = 1.
if exo_apres is Ellipsis or exo_apres is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_apres)-exo_w.item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_w = torch.tensor(0.,requires_grad=True)
exo_optimiseur = torch.optim.SGD([exo_w],lr=.5)
((exo_w-2.)**2).backward()
exo_optimiseur.step()
exo_perte_apres = 0.
if exo_perte_apres is Ellipsis or exo_perte_apres is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_perte_apres)-((exo_w-2.)**2).item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [24]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()